In [ ]:
from ffc import *
# =====================================================================
# JUPYTER TEST EXECUTION (USING REAL HL7 JSON)
# =====================================================================
if __name__ == "__main__":
    # To run this, you need the official HL7 profiles-resources.json files in your directory.
    r4_bundle_path = "profiles-resources-r4.json"
    r5_bundle_path = "profiles-resources-r5.json"
    
    if os.path.exists(r4_bundle_path) and os.path.exists(r5_bundle_path):
        print("Loading official HL7 R4 and R5 Bundles...")
        
        with open(r4_bundle_path, 'r', encoding='utf-8') as f:
            r4_bundle = json.load(f)
            
        with open(r5_bundle_path, 'r', encoding='utf-8') as f:
            r5_bundle = json.load(f)
            
        try:
            r4_elements = extract_structure_definition(r4_bundle, "Observation")
            r5_elements = extract_structure_definition(r5_bundle, "Observation")
            
            schemas_by_version = [("R4", r4_elements), ("R5", r5_elements)]
            
            print("Compiling Version-Agnostic FastFHIR Observation Domain...\n")
            master_blocks = merge_fhir_versions(schemas_by_version, "Observation")
            hpp_code, cpp_code = emit_domain_file(master_blocks, "Observation", ["R4", "R5"])
            
            with open("FF_Observation.hpp", "w") as hpp_file: hpp_file.write(hpp_code)
            with open("FF_Observation.cpp", "w") as cpp_file: cpp_file.write(cpp_code)
            
            print(f"✅ Successfully wrote to disk: FF_Observation.hpp and FF_Observation.cpp")
            
        except Exception as e:
            print(f"Compilation Error: {e}")
            
    else:
        print(f"⚠️ Missing HL7 Files. Please download '{r4_bundle_path}' and '{r5_bundle_path}' into this directory.")

In [ ]:
from ffd import generate_dictionary_files
# =====================================================================
# JUPYTER TEST EXECUTION
# =====================================================================
if __name__ == "__main__":
    # Mock FHIR ValueSets
    mock_valuesets = [
        {
            "compose": {
                "include": [
                    {
                        "system": "http://hl7.org/fhir/observation-status",
                        "concept": [
                            {"code": "registered"},
                            {"code": "preliminary"},
                            {"code": "final"},
                            {"code": "amended"}
                        ]
                    },
                    {
                        "system": "http://loinc.org",
                        "concept": [
                            {"code": "15074-8"} # Example LOINC code (Glucose)
                        ]
                    }
                ]
            }
        }
    ]

    print("Compiling FastFHIR Dictionary...\n")
    hpp_code, cpp_code = generate_dictionary_files(mock_valuesets)
    
    # Write to Disk
    hpp_filename = "FF_Dictionary.hpp"
    cpp_filename = "FF_Dictionary.cpp"
    
    with open(hpp_filename, "w") as hpp_file: hpp_file.write(hpp_code)
    with open(cpp_filename, "w") as cpp_file: cpp_file.write(cpp_code)
        
    print(f"✅ Successfully wrote to disk: {hpp_filename} and {cpp_filename}")